# 🔑 Keyhole — Fraud Spike Detector: Model Training

**Razorpay AI Buildathon — Track 02: AI Risk Manager**

Trains an Isolation Forest on the [Kaggle Credit Card Fraud dataset](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) and exports a calibrated artifact for the real-time Keyhole pipeline.

**Honest evaluation design:**
- **Train split**: first 80% of dataset time span, *fraud rows excluded* (anomaly detection convention)
- **Held-out test split**: last 20% — never seen in training, labels used only for metrics
- **Threshold**: calibrated on train scores to a target false-positive rate

**How to use on Kaggle:**
1. New Notebook → Add Data → search `creditcardfraud` (mlg-ulb)
2. Run all cells
3. Download `keyhole_model.joblib` from the output pane → place in `Keyhole/models/isolation_forest.joblib`
4. `docker compose up --build` — the app skips training and starts streaming

In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

DATA = "/kaggle/input/creditcardfraud/creditcard.csv"  # attached dataset
TIME_SPAN = 172_792.0   # dataset covers 48h (seconds)
TRAIN_FRACTION = 0.8    # first 80% of time = train, last 20% = held-out live stream
TARGET_FPR = 0.005      # operating point: 0.5% of normal txns flagged

In [ ]:
df = pd.read_csv(DATA)
df["AmountLog"] = np.log1p(df["Amount"])  # tame heavy-tailed amounts

features = [f"V{i}" for i in range(1, 29)] + ["AmountLog"]
cutoff = TIME_SPAN * TRAIN_FRACTION

train = df[(df["Time"] < cutoff) & (df["Class"] == 0)]  # normal-only
test = df[df["Time"] >= cutoff]                        # held-out, labels kept

print(f"train (normal only): {len(train):,} | held-out: {len(test):,} (frauds: {test['Class'].sum()})")
X_train, X_test, y_test = train[features].values, test[features].values, test["Class"].values

In [ ]:
scaler = StandardScaler().fit(X_train)
model = IsolationForest(n_estimators=200, contamination=0.002, random_state=42, n_jobs=-1)
model.fit(scaler.transform(X_train))

# Calibrate threshold on train scores to hit TARGET_FPR
train_scores = model.decision_function(scaler.transform(X_train))
threshold = float(np.percentile(train_scores, 100 * TARGET_FPR))
print(f"calibrated threshold: {threshold:.4f} (target FPR {TARGET_FPR:.1%})")

In [ ]:
# Held-out evaluation — the honest numbers
scores = model.decision_function(scaler.transform(X_test))
pred = scores < threshold

tp = int((pred & (y_test == 1)).sum()); fp = int((pred & (y_test == 0)).sum())
fn = int((~pred & (y_test == 1)).sum()); tn = int((~pred & (y_test == 0)).sum())
precision = tp / (tp + fp); recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)
fpr = fp / (fp + tn)

print(f"TP={tp} FP={fp} FN={fn} TN={tn}")
print(f"precision={precision:.3f}  recall={recall:.3f}  f1={f1:.3f}  fpr={fpr:.4f}")

In [ ]:
# Operating-point sweep — documented trade-off for the pitch
rows = []
for fpr_target in [0.001, 0.005, 0.01, 0.02, 0.05]:
    t = float(np.percentile(train_scores, 100 * fpr_target))
    p_ = scores < t
    tp_ = int((p_ & (y_test == 1)).sum()); fp_ = int((p_ & (y_test == 0)).sum())
    prec = tp_ / max(tp_ + fp_, 1); rec = tp_ / max((y_test == 1).sum(), 1)
    f1_ = 2 * prec * rec / max(prec + rec, 1e-9)
    rows.append({"target_fpr": fpr_target, "precision": round(prec, 3),
                 "recall": round(rec, 3), "f1": round(f1_, 3)})
pd.DataFrame(rows)

In [ ]:
# Export calibrated artifact — download this and drop into Keyhole/models/
joblib.dump({"model": model, "scaler": scaler, "threshold": threshold},
            "/kaggle/working/keyhole_model.joblib")
print("Saved /kaggle/working/keyhole_model.joblib → rename to isolation_forest.joblib")